# Triangular register indexing

Copyright 2024 Google LLC. Adapted under Apache-2.0 from locked Qualtran v0.7.0, commit `bcb17bb25360ed68ce3bc084645db3e929799389`.


In [ ]:
from collections import Counter
from collections.abc import Callable
from typing import cast

from IPython import display as notebook_display

from examples.algorithms.classical_contiguous_index_values import (
    main as execute_example,
)
from flagquantum.compiler._component_trace.call_graph import (
    SymbolicCallGraph,
    symbolic_call_graph,
)
from flagquantum.compiler._component_trace.resource_generalization import (
    FixedTCall,
    ResourceCall,
)
from flagquantum.compiler._component_trace.resource_model import ComponentCost
from flagquantum.compiler.registers.sorting.contiguous_index import (
    contiguous_index_columns_ir,
    contiguous_index_columns_workspace_size,
    source_contiguous_index_cost,
)
from flagquantum.core.ir.program.register_types import unsigned_register_type
from flagquantum.core.ir.program.registers import Register
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.drawer.register_graph_table import signature_instance_table
from flagquantum.drawer.register_score import RegisterStep

# Optional IPython functions enter through the presentation boundary.
SVG = cast(Callable[[bytes], object], notebook_display.SVG)
display = cast(Callable[[object], None], notebook_display.display)


def title(call: ResourceCall) -> str:
    return "Triangular index" if isinstance(call, ComponentCost) else "Toffoli"


## Contiguous triangular index
Build a contiguous register s from mu and nu.

$$
    s = \nu (\nu + 1) / 2 + \mu
$$

Assuming nu is zero indexed (in contrast to the THC paper which assumes 1,
hence the slightly different formula).

#### Parameters
 - `bitsize`: number of bits for mu and nu registers.
 - `s_bitsize`: Number of bits for contiguous register. 

#### Registers
 - `mu`: input register
 - `nu`: input register
 - `s`: output contiguous register 

#### References
 - [Even more efficient quantum computations of chemistry through tensor hypercontraction](https://arxiv.org/pdf/2011.03494.pdf). Eq. 29.


In [ ]:
# The resource declaration and physical emitter remain separate owners.


### Example Instances

In [ ]:
to_contg_index = source_contiguous_index_cost(4, 8)
index_signature = Signature(
    (
        Register("mu", unsigned_register_type(4)),
        Register("nu", unsigned_register_type(4)),
        Register("s", unsigned_register_type(8)),
    )
)


#### Graphical Signature

In [ ]:
ports = tuple(reg.name for reg in index_signature)
table = signature_instance_table(
    index_signature,
    RegisterStep("Triangular index (4, 8)", ports, ports),
    {name: f"p{i}" for i, name in enumerate(ports)},
)
table = table.replace(
    "</TABLE>",
    '<TR><TD COLSPAN="2">mu=uint[4], nu=uint[4], s=uint[8]</TD></TR></TABLE>',
)
display(
    SVG(
        render_call_graph_svg(
            "digraph signature { n [shape=plain,label=<" + table + ">]; }"
        )
    )
)


### Call Graph

In [ ]:
root: ResourceCall = to_contg_index
index_graph: SymbolicCallGraph[ResourceCall] = symbolic_call_graph(
    root, {root: {FixedTCall("toffoli"): to_contg_index.gates.toffoli}}, max_depth=1
)
display(
    SVG(
        render_call_graph_svg(
            export_call_graph_dot(
                index_graph.root, index_graph.direct_calls, title=title
            )
        )
    )
)
display(index_graph.leaf_counts)


## Native execution and independent accounts

The source root has a declared Toffoli model and no gate decomposition. Its raw classical callback computes the plus polynomial above, while the source score glyph uses a minus polynomial. Those source inconsistencies remain in the full file audit.

The native operation XORs the computed value into an occupied target and restores its clean work. The existing one-bit append example verifies complete complex amplitudes with a spectator and restores the input on a second append. Its measured primitive report is separate from the original root model. Below, the original four/eight-bit dimensions are also emitted and counted without dense-state allocation.


In [ ]:
execute_example()
work_width = contiguous_index_columns_workspace_size(4)
gates = contiguous_index_columns_ir(
    tuple(range(4)),
    tuple(range(4, 8)),
    tuple(range(8, 16)),
    tuple(range(16, 16 + work_width)),
)
measured_primitives = dict(Counter(g.name for g in gates))
display(measured_primitives)
display(to_contg_index.gates)
